## 01. Data Structure Check

### 1. `data/raw/` 폴더 열어서 어떤 파일이 있는지 확인

In [ ]:
# 파일과 폴더 경로를 다루기 위한 Path를 불러옵니다.
from pathlib import Path

# 현재 노트북 위치를 기준으로 원본 데이터 폴더 경로를 지정합니다.
raw_data_path = Path("../data/raw")

# data/raw 폴더 바로 아래에 있는 파일과 폴더를 이름순으로 하나씩 가져옵니다.
for path in sorted(raw_data_path.iterdir()):
    # 가져온 파일 또는 폴더의 전체 경로가 아니라 이름만 출력합니다.
    print(path.name)

# data/raw 안에서 확인할 대표 폴더 이름 3개를 리스트로 저장합니다.
folder_names = ["test_images", "train_annotations", "train_images"]

# 대표 폴더 이름을 하나씩 꺼내서 반복합니다.
for folder_name in folder_names:
    # 현재 확인할 폴더의 전체 경로를 만듭니다.
    folder_path = raw_data_path / folder_name

    # 어떤 폴더를 확인하는 중인지 보기 좋게 출력합니다.
    print(f"\n[{folder_name}]")

    # 현재 폴더 바로 아래에 있는 파일과 폴더를 이름순으로 정렬합니다.
    inner_paths = sorted(folder_path.iterdir())

    # 정렬된 항목 중 앞에서 3개만 꺼내서 반복합니다.
    for inner_path in inner_paths[:3]:
        # 항목이 폴더인지 파일인지 구분하기 위한 글자를 만듭니다.
        path_type = "folder" if inner_path.is_dir() else "file"

        # 항목의 종류와 이름을 출력합니다.
        print(f"{path_type}: {inner_path.name}")


### 2. 어노테이션 폴더 상세확인

In [ ]:
annotation_folder = "K-001900-016548-019607-029451_json"
annotation_folder_path = raw_data_path / "train_annotations" / annotation_folder

# 현재 확인할 annotation 폴더 경로를 출력합니다.
print(annotation_folder_path)

# annotation 폴더 안에 있는 모든 파일을 하위 폴더까지 포함해서 이름순으로 가져옵니다.
annotation_files = sorted(path for path in annotation_folder_path.rglob("*") if path.is_file())

# 찾은 파일 개수를 출력합니다.
print(f"file count: {len(annotation_files)}")

# annotation 폴더 안에서 찾은 파일들을 하나씩 반복합니다.
for annotation_file in annotation_files:
    # 전체 경로 대신 annotation 폴더 기준의 상대 경로로 출력합니다.
    print(annotation_file.relative_to(annotation_folder_path))

### 3. annotation 파일(JSON 또는 CSV) 열어서 컬럼/키 목록 파악

In [ ]:
# 열어볼 JSON 파일 경로를 지정합니다.
json_file_path = raw_data_path / "train_annotations" / "K-001900-016548-019607-029451_json" / "K-001900" / "K-001900-016548-019607-029451_0_2_0_2_70_000_200.json"

# JSON 파일을 텍스트 원본 그대로 엽니다.
with open(json_file_path, "r", encoding="utf-8") as file:
    # 파일 전체 내용을 문자열로 읽어옵니다.
    json_text = file.read()

# 읽어온 JSON 원본 내용을 그대로 출력합니다.
print(json_text)


### 4. 이미지 1장과 annotation 1개를 매핑해서 bbox 좌표 확인

In [ ]:
import json
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# 앞 셀에서 만든 annotation_folder_path를 기준으로 확인할 JSON 파일 경로를 짧게 지정합니다.
json_file_path = annotation_folder_path / "K-001900" / "K-001900-016548-019607-029451_0_2_0_2_70_000_200.json"
with open(json_file_path, "r", encoding="utf-8") as file:
    annotation_data = json.load(file)

# JSON 안의 첫 번째 이미지 정보와 첫 번째 annotation 정보를 가져옵니다.
image_info = annotation_data["images"][0]
annotation_info = annotation_data["annotations"][0]
category_info = annotation_data["categories"][0]

# 이미지 파일명, bbox 좌표, 카테고리 이름을 각각 꺼냅니다.
image_file_name = image_info["file_name"]
bbox = annotation_info["bbox"]
category_name = category_info["name"]

# train_images 폴더에서 JSON에 적힌 이미지 파일명과 같은 이미지를 찾습니다.
image_file_path = next((raw_data_path / "train_images").rglob(image_file_name))
image = Image.open(image_file_path)

# bbox는 [x, y, width, height] 형식이므로 각각 나눠서 저장합니다.
x, y, width, height = bbox

# 이미지 위에 bbox 사각형과 카테고리 이름을 그립니다.
fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(image)
ax.add_patch(patches.Rectangle((x, y), width, height, linewidth=2, edgecolor="red", facecolor="none"))
ax.text(x, y - 8, category_name, color="red", fontsize=12, backgroundcolor="white")
ax.axis("off")
plt.show()

### 5. 클래스 수와 종류 확인

In [ ]:
# JSON 파일을 읽고 클래스 정보를 모으기 위한 준비입니다.
import json

# 전체 annotation JSON 파일 목록을 가져오고, 클래스 ID와 이름을 저장할 딕셔너리를 만듭니다.
class_id_to_name = {}
annotation_json_files = sorted((raw_data_path / "train_annotations").rglob("*.json"))

# 모든 annotation JSON 파일을 열어서 categories 안의 클래스 정보를 수집합니다.
for annotation_json_file in annotation_json_files:
    with open(annotation_json_file, "r", encoding="utf-8") as file:
        annotation_data = json.load(file)
    for category in annotation_data["categories"]:
        class_id_to_name[category["id"]] = category["name"]

# 중복을 제거한 클래스 수를 출력합니다.
print(f"클래스 수: {len(class_id_to_name)}")

# 클래스 ID 기준으로 정렬한 뒤, 클래스 ID와 클래스 이름을 전부 출력합니다.
for class_id, class_name in sorted(class_id_to_name.items()):
    print(f"{class_id}: {class_name}")

### 6. bbox 형식 확인

현재 annotation의 `bbox` 값은 `[x, y, width, height]` 형식입니다.

예시:

```text
bbox = [644, 845, 189, 190]
```

각 값의 의미는 다음과 같습니다.

- `x`: bbox의 왼쪽 위 x 좌표
- `y`: bbox의 왼쪽 위 y 좌표
- `width`: bbox의 가로 길이
- `height`: bbox의 세로 길이

따라서 이 데이터셋의 bbox는 `xyxy`가 아니라 `xywh` 형식입니다.

`xyxy` 형식이 필요하면 다음처럼 변환할 수 있습니다.

```python
x_min = x
y_min = y
x_max = x + width
y_max = y + height
```

### 7. 파일 구조 tree

현재 확인한 `data/raw/` 폴더 구조는 다음과 같습니다.

```text
data/
└── raw/
    ├── test_images/
    │   ├── 1.png
    │   ├── 10.png
    │   └── 100.png
    ├── train_images/
    │   ├── K-001900-016548-019607-029451_0_2_0_2_70_000_200.png
    │   ├── K-001900-016548-019607-029451_0_2_0_2_75_000_200.png
    │   └── K-001900-016548-019607-029451_0_2_0_2_90_000_200.png
    └── train_annotations/
        ├── K-001900-016548-019607-029451_json/
        │   ├── K-001900/
        │   │   ├── K-001900-016548-019607-029451_0_2_0_2_70_000_200.json
        │   │   ├── K-001900-016548-019607-029451_0_2_0_2_75_000_200.json
        │   │   └── K-001900-016548-019607-029451_0_2_0_2_90_000_200.json
        │   ├── K-016548/
        │   ├── K-019607/
        │   └── K-029451/
        ├── K-001900-016548-019607-033009_json/
        └── K-001900-016548-021771-027926_json/
```

`train_images/`에는 실제 학습 이미지가 있고, `train_annotations/`에는 이미지와 연결되는 JSON annotation 파일이 폴더 단위로 저장되어 있습니다.

확인한 특징은 다음과 같습니다.

1. `train_images/`에는 같은 이름 패턴의 이미지가 3장씩 있는 것으로 보입니다. 파일명 기준으로 `70`, `75`, `90`처럼 카메라 위도 3종류가 포함되어 있습니다.
2. `train_annotations/`에는 각 알약 종류별로 annotation JSON이 3개씩 있는 것으로 보입니다. 이미지와 마찬가지로 카메라 위도 3종류에 대응합니다.

### 8. 어노테이션 컬럼 목록

| 구분 | 항목명 | 타입 | 필수여부 | 설명 |
|---|---|---|---|---|
| images | images | Object | M | 약제 이미지정보 |
| images | images[].id | Number | M | 약제 이미지식별자 |
| images | images[].width | Number | M | 약제 이미지너비 |
| images | images[].height | Number | M | 약제 이미지높이 |
| images | images[].file_name | String | M | 약제 이미지파일명 |
| images | images[].drug_N | String | M | 알약코드 |
| images | images[].drug_S | String | M | 알약상태 |
| images | images[].back_color | String | M | 촬영배경 |
| images | images[].drug_dir | String | M | 알약방향 |
| images | images[].light_color | String | M | 촬영조명 |
| images | images[].camera_la | Number | M | 카메라위도 |
| images | images[].camera_lo | Number | M | 카메라경도 |
| images | images[].size | Number | M | 이미지 사이즈 |
| images | images[].dl_idx | String | M | 알약 식별자 |
| images | images[].dl_mapping_code | String | M | 제품코드 |
| images | images[].dl_name | String | M | 제품명 |
| images | images[].dl_name_en | String | O | 제품명(영어) |
| images | images[].img_key | String | M | 이미지 링크 |
| images | images[].dl_material | String | M | 성분명 |
| images | images[].dl_material_en | String | O | 성분명(영어) |
| images | images[].dl_custom_shape | String | M | 제조 모양 |
| images | images[].dl_company | String | M | 제조사명 |
| images | images[].dl_company_en | String | O | 제조사명(영어) |
| images | images[].di_company_mf | String | M | 위탁제조사명 |
| images | images[].di_company_mf_en | String | O | 위탁제조사명(영어) |
| images | images[].item_seq | Number | M | 품목기준코드 |
| images | images[].di_item_permit_date | Date | O | 허가일자 |
| images | images[].di_class_no | String | M | 약품 분류 |
| images | images[].di_etc_otc_code | String | M | 전문의약품/일반의약품 |
| images | images[].di_edi_code | String | M | EDI 코드 |
| images | images[].chart | String | M | 알약 제형 |
| images | images[].drug_shape | String | M | 알약 모양 |
| images | images[].thick | Number | M | 알약 두께 |
| images | images[].leng_long | Number | M | 알약 장축 |
| images | images[].leng_short | Number | M | 알약 단축 |
| images | images[].print_front | String | C | 식별문자_앞 |
| images | images[].print_back | String | C | 식별문자_뒤 |
| images | images[].color_class1 | String | M | 색상 1 |
| images | images[].color_class2 | String | O | 색상 2 |
| images | images[].line_front | String | M | 알약 앞면분할선 여부 |
| images | images[].line_back | String | M | 알약 뒷면분할선 여부 |
| images | images[].img_regist_ts | Date | O | 약학정보원 이미지 생성일 |
| images | images[].form_code_name | String | M | 정제 분류명 |
| images | images[].mark_code_front_anal | String | M | 알약 앞면마크 형태 |
| images | images[].mark_code_back_anal | String | M | 알약 뒷면마크 형태 |
| images | images[].mark_code_front_img | String | M | 알약 앞면마크 이미지 |
| images | images[].mark_code_back_img | String | M | 알약 뒷면마크 이미지 |
| images | images[].mark_code_front | String | M | 알약 앞면마크 코드 |
| images | images[].mark_code_back | String | M | 알약 뒷면마크 코드 |
| images | images[].change_date | Date | O | 변경일자 |
| type | type | type | O | json 타입 |
| annotations | annotations | Object | M | 라벨링정보 |
| annotations | annotations[].area | Number | M | 바운딩박스 면적 |
| annotations | annotations[].iscrowd | Number | O | 평가 분류 |
| annotations | annotations[].bbox | List | M | bbox 좌표 |
| annotations | annotations[].category_id | Number | M | category 아이디 |
| annotations | annotations[].ignore | Number | O | 무시 여부 |
| annotations | annotations[].segmentation | List | O | 라벨링바운딩 박스 |
| annotations | annotations[].image_id | Number | M | 이미지 아이디 |
| annotations | annotations[].id | Number | M | 어노테이션 아이디 |
| categories | categories | Object | M | 라이선스 |
| categories | categories[].supercategory | String | M | 슈퍼 카테고리 |
| categories | categories[].id | Number | M | 카테고리 아이디 |
| categories | categories[].name | String | C | 카테고리 명 |